# Append_File Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Gentle append.** `"a"` parks the cursor at the end of the file and leaves everything before it untouched.

In [ ]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

with open("sample_data/diary.txt", "w", encoding="utf-8") as f:
    f.write("Day 1: started file handling\n")

with open("sample_data/diary.txt", "a", encoding="utf-8") as f:
    f.write("Day 2: learned append mode\n")

print(Path("sample_data/diary.txt").read_text(encoding="utf-8"))

**2. Append creates missing files too.** Like `"w"`, append auto-creates a missing file — but it preserves existing content instead of wiping it.

In [ ]:
from pathlib import Path

fresh = "sample_data/fresh_log.txt"

with open(fresh, "a", encoding="utf-8") as f:
    f.write("first entry ever\n")

print(Path(fresh).exists())                  # True
print(Path(fresh).read_text(encoding="utf-8"))

# "w" also auto-creates - but it WIPES any existing file first;
# "a" creates AND preserves.

**3. The welded note.** Appends without `"\\n"` weld together mid-line; one extra `"\\n"` repairs the damage.

In [ ]:
from pathlib import Path

sticky = "sample_data/sticky.txt"

with open(sticky, "a", encoding="utf-8") as f:
    f.write("buy milk")
with open(sticky, "a", encoding="utf-8") as f:
    f.write("call the plumber")

print(repr(Path(sticky).read_text(encoding="utf-8")))
# 'buy milkcall the plumber'  <- welded!

with open(sticky, "a", encoding="utf-8") as f:
    f.write(" -- oops, forgot the newline!\n")
    f.write("this entry ends cleanly\n")

for line in Path(sticky).read_text(encoding="utf-8").splitlines():
    print("-", line)

## Part 2 — Practice

**4. Three runs, one log.** Each `with` block targets the current end of file, so separate opens weave one continuous timeline.

In [ ]:
from pathlib import Path

sessions = [
    "2026-08-24 | studied loops",
    "2026-08-25 | studied functions",
    "2026-08-26 | studied file handling",
]

for session in sessions:                     # each block = one 'program run'
    with open("sample_data/study_log.txt", "a", encoding="utf-8") as f:
        f.write(session + "\n")

print(Path("sample_data/study_log.txt").read_text(encoding="utf-8"))

# Nothing was harmed between opens: "a" always aims at the current END,
# so yesterday's entries survive today's.

**5. The invisible entry.** In `"a+"` the cursor starts at the end, so `read()` sees nothing until `seek(0)` rewinds it.

In [ ]:
with open("sample_data/cursor.txt", "a+", encoding="utf-8") as f:
    f.write("entry written at the end\n")
    print("cursor sits at:", f.tell())
    print("read right now:", repr(f.read()))   # '' - cursor was at the END

    f.seek(0)                                  # rewind to byte 0
    print("after seek(0):")
    print(f.read())

# The cursor is the file object's position marker: append modes park it
# at the end, so a fresh read finds nothing behind it.

**6. `x` marks the exclusive.** `"x"` refuses to touch an existing file — a loud `FileExistsError` beats silent data loss.

In [ ]:
from pathlib import Path

submission = "sample_data/exam_submission.txt"

for attempt in (1, 2):
    try:
        with open(submission, "x", encoding="utf-8") as f:
            f.write("final answers, locked forever\n")
        print(f"Attempt {attempt}: created.")
    except FileExistsError:
        print(f"Attempt {attempt}: refused - file already exists.")

print(Path(submission).read_text(encoding="utf-8").strip())
# Attempt 1: created.
# Attempt 2: refused - file already exists.
# final answers, locked forever

## Part 3 — Challenge

**7. A tiny event logger.** Wrapping the append logic in a function keeps the formatting in one place; the file is streamed back numbered.

In [ ]:
STAMP = "2026-08-26"

def log_event(message):
    """Append one dated event to the log file."""
    with open("sample_data/events.log", "a", encoding="utf-8") as f:
        f.write(f"[{STAMP}] {message}\n")

log_event("started the append lesson")
log_event("wrote a logger function")
log_event("logged my own victory")

with open("sample_data/events.log", encoding="utf-8") as f:
    for number, line in enumerate(f, start=1):
        print(number, "->", line.rstrip("\n"))

**8. High-score table, three modes.** `"a"` grows the history and `"w"` reshapes it — and `"x"` is the mode that never risks an overwrite.

In [ ]:
from pathlib import Path

scores_path = "sample_data/scores.txt"

# 1) "w": reset the table so reruns stay tidy.
with open(scores_path, "w", encoding="utf-8") as f:
    f.write("Sarah 120\n")
    f.write("Tanvir 95\n")

# 2) "a": new results join the end, history intact.
with open(scores_path, "a", encoding="utf-8") as f:
    f.write("Amina 150\n")
    f.write("Rafi 60\n")

# 3) "w" again: reshape the file into a ranked leaderboard.
with open(scores_path, encoding="utf-8") as f:
    entries = [line.split() for line in f]
ranked = sorted(entries, key=lambda pair: int(pair[1]), reverse=True)
with open(scores_path, "w", encoding="utf-8") as f:
    for name, score in ranked:
        f.write(f"{name} {score}\n")

print(Path(scores_path).read_text(encoding="utf-8"))
# Amina 150
# Sarah 120
# Tanvir 95
# Rafi 60

# "x" mode protects snapshots: it raises FileExistsError instead of
# ever touching an existing file.